# FineTuning SLM QLoRA JurisEval

**FineTuning SLM QLoRA JurisEval** is a Proof of Concept (PoC) focused on the end-to-end cycle of adaptation, specialization, and evaluation (*fine-tuning & evaluation*) of Small Language Models (SLMs) tailored for the *Brazilian legal domain*. The pipeline automates efficient parameter training via **QLoRA (4-bit NF4 with the PagedAdamW optimizer)** using the `TRL/PEFT` framework under accessible hardware constraints (such as T4 GPUs), targeting a 1.5B parameter SLM (`Qwen2.5-1.5B-Instruct) for complex tasks in **Criminal Law, Cyber Law, and Data Protection**. Following training, the script executes a **customized multidimensional benchmark (JurisEval)** inspired by the three levels of *LawBench* (Memorization, Comprehension, and Application), rigorously measuring semantic quality metrics (**ROUGE-L**), strict technical vocabulary retention (*Keyword Recall*), memory consumption (**VRAM in MB**), and inference latency.

## Technical Highlights:
* **4-bit Optimized Training:** A clean implementation of QLoRA using `BitsAndBytesConfig` and `prepare_model_for_kbit_training`, enabling fine-tuning on modest GPUs.
* **Alignment and Formatting via Chat Template:** Standardization using Qwen instruction tokens (`<|im_start|>`, `<|im_end|>`) to enforce domain-specific legal persona behavior.

**Multidimensional Legal Evaluation (`JurisEval`):**

* **Textual Quality:** Similarity validation using *ROUGE-L* scored against reference target responses.
* **Doctrinal/Legal Compliance:** Automated verification of mandatory technical term retrieval (*Keyword Recall* focusing on statutes, ADPFs, and legal doctrines).
* **Computational Efficiency:** Dynamic runtime monitoring of peak VRAM usage and per-inference latency.

## Disclaimer

This Proof of Concept (PoC) exhibits critical structural limitations due to its reliance on a single-example training dataset, resulting in severe overfitting and rote memorization that leads to degenerate, repetitive output loops and a sharp performance decline in comprehension tasks where *ROUGE-L* plummets to *0.2537*. Furthermore, the *JurisEval* evaluation is statistically insufficient because it covers only three test cases and relies solely on lexical and syntactic metrics (*ROUGE-L* and *Keyword Recall*) that fail to assess true logical coherence or legal validity, alongside runtime dependency conflicts between Python packages (`pyarrow` and `cudf`). To address these shortcomings and advance the project toward production readiness or advanced research, it is essential to **expand the dataset with thousands of diverse, real-world legal instructions**, implement repetition penalties during text generation, containerize the environment using Docker to ensure technical reproducibility, and **strengthen JurisEval by incorporating domain-expert human evaluations, an LLM-as-a-judge framework, and robust literature benchmarks like *LawBench* and *LexRAG***.

In [1]:
!pip install -q -U torch torchvision torchaudio transformers datasets accelerate peft trl bitsandbytes rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [11]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. 4-bit QLoRA configuration (FP16 computation)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16 # bitsandbytes manages FP16 computation
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# 2. Loading the Model, Forcing FP16 (Overrides native BF16)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    dtype=torch.float16, # Converts residual tensors to float16
    device_map="auto"
)

# Preparing the model for k-bit
model = prepare_model_for_kbit_training(model)

# 3. LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 4. Dataset and Formatting
data = [
    {
        "instruction": "Qual a exigência constitucional para o uso de softwares de intrusão remota na investigação criminal?",
        "output": "Exige-se autorização judicial prévia e individualizada (reserva de jurisdição qualificada), relatório de impacto prévio e fiscalização técnica independente, sendo vedado o uso para inteligência preventiva desvinculada de investigação individualizada (ADPF 1.143 e Anteprojeto da LGPD-Penal)."
    }
]
dataset = Dataset.from_list(data)

def format_prompts(example):
    return (
        f"<|im_start|>system\nVocê é um assistente jurídico especializado em Direito Penal Digital e Proteção de Dados.<|im_end|>\n"
        f"<|im_start|>user\n{example['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['output']}<|im_end|>"
    )

# 5. Training Arguments (Without AMP GradScaler)
training_args = TrainingArguments(
    output_dir="./results_qlora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=50,
    fp16=False, # Disable the manual GradScaler from PyTorch AMP
    bf16=False, # Disable BFloat16, which is incompatible with T4
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}, # Silences PyTorch reentrancy warning
    optim="paged_adamw_8bit",
    save_strategy="no"
)

# 6. Execution of SFTTrainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=format_prompts,
    args=training_args
)

trainer.train()
trainer.model.save_pretrained("./legal_qlora_adapter")
print("✅ Fine-tuning with QLoRA (4-bit NF4) successfully executed on a T4 GPU!")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Applying formatting function to train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Step,Training Loss
10,1.162560
20,0.004251
30,0.000632
40,0.000400
50,0.000356


✅ Fine-tuning with QLoRA (4-bit NF4) successfully executed on a T4 GPU!


In [13]:
import time
import torch
import pandas as pd
from rouge_score import rouge_scorer

# 1. Function for Calculating Keyword Accuracy
def check_keyword_accuracy(response_text, mandatory_keywords):
    response_lower = response_text.lower()
    found_keywords = [kw for kw in mandatory_keywords if kw.lower() in response_lower]

    hits = len(found_keywords)
    total_mandatory = len(mandatory_keywords)
    recall = hits / total_mandatory if total_mandatory > 0 else 0.0

    return {
        "keyword_recall": round(recall, 4),
        "found_keywords": found_keywords,
        "missing_keywords": [kw for kw in mandatory_keywords if kw.lower() not in response_lower]
    }

# 2. Corrected Inference Function
def evaluate_inference_v2(model, tokenizer, prompt, target_text, mandatory_keywords):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=250, temperature=0.1)

    latency_ms = (time.time() - start_time) * 1000
    peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)

    # Index [1] accesses the number of tokens in the input sequence
    prompt_len = inputs.input_ids.shape[1]

    # Correct slicing of the first sequence (index 0) from the end of the prompt
    generated_tokens = outputs[0][prompt_len:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # 1. ROUGE-L Metrics
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_l = scorer.score(target_text, response)['rougeL'].fmeasure

    # 2. Recall Metrics (Keyword Recall)
    kw_metrics = check_keyword_accuracy(response, mandatory_keywords)

    return {
        "response": response,
        "latency_ms": round(latency_ms, 2),
        "vram_mb": round(peak_vram_mb, 2),
        "rouge_l": round(rouge_l, 4),
        "keyword_recall": kw_metrics["keyword_recall"],
        "missing_keywords": kw_metrics["missing_keywords"]
    }

# 3. Test Cases Covering the 3 Levels of LawBench
test_cases = [
    {
        "nivel": "1. Memorização",
        "instruction": "Qual a exigência constitucional para o uso de softwares de intrusão remota na investigação criminal?",
        "target": "Exige-se autorização judicial prévia e individualizada (reserva de jurisdição qualificada), relatório de impacto prévio e fiscalização técnica independente, sendo vedado o uso para inteligência preventiva desvinculada de investigação individualizada (ADPF 1.143 e Anteprojeto da LGPD-Penal).",
        "keywords": ["autorização judicial", "reserva de jurisdição", "relatório de impacto", "ADPF 1.143", "LGPD-Penal"]
    },
    {
        "nivel": "2. Compreensão",
        "instruction": "A polícia civil pode instalar um software espião preventivamente no celular de um suspeito sem pedir ao juiz?",
        "target": "Não. O uso exige autorização judicial prévia e individualizada (reserva de jurisdição qualificada), além de relatório de impacto prévio e fiscalização técnica independente, sendo vedada a inteligência preventiva desvinculada de investigação (ADPF 1.143 e Anteprojeto da LGPD-Penal).",
        "keywords": ["não", "autorização judicial", "inteligência preventiva", "ADPF 1.143", "LGPD-Penal"]
    },
    {
        "nivel": "3. Aplicação",
        "instruction": "Em uma operação, a autoridade policial extraiu dados de um celular apreendido sem apresentar relatório de impacto prévio à autoridade judicial. A prova é válida?",
        "target": "A prova é ilícita. A ausência de relatório de impacto prévio e de fiscalização técnica independente descumpre os requisitos formais de validade exigidos pela ADPF 1.143 e pela LGPD-Penal.",
        "keywords": ["ilícita", "relatório de impacto", "fiscalização técnica", "ADPF 1.143", "LGPD-Penal"]
    }
]

# 4. Benchmark Execution
resultados_bench = []
model.eval()

print("🚀 Running tests across the 3 levels of LawBench with vocabulary analysis...")

for case in test_cases:
    prompt = (
        f"<|im_start|>system\nVocê é um assistente jurídico especializado em Direito Penal Digital e Proteção de Dados.<|im_end|>\n"
        f"<|im_start|>user\n{case['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    metrics = evaluate_inference_v2(
        model, tokenizer, prompt, case["target"], case["keywords"]
    )

    resultados_bench.append({
        "Level (LawBench)": case["nivel"],
        "Latency (ms)": metrics["latency_ms"],
        "Peak VRAM (MB)": metrics["vram_mb"],
        "ROUGE-L": metrics["rouge_l"],
        "Required Vocabulary (%)": f"{metrics['keyword_recall'] * 100:.1f}%",
        "Missing Terms": ", ".join(metrics["missing_keywords"]) if metrics["missing_keywords"] else "Nenhum",
        "Generated Response": metrics["response"]
    })

# 5. Display of the Final Table
df_resultados = pd.DataFrame(resultados_bench)
pd.set_option('display.max_colwidth', None)

print("\n✅ Multidimensional Legal Assessment Table:\n")
display(df_resultados)

🚀 Running tests across the 3 levels of LawBench with vocabulary analysis...

✅ Multidimensional Legal Assessment Table:



,Level (LawBench),Latency (ms),Peak VRAM (MB),ROUGE-L,Required Vocabulary (%),Missing Terms,Generated Response
0,1. Memorização,8969.76,3408.38,1.0000,100.0%,Nenhum,"Exige-se autorização judicial prévia e individualizada (reserva de jurisdição qualificada), relatório de impacto prévio e fiscalização técnica independente, sendo vedado o uso para inteligência preventiva desvinculada de investigação individualizada (ADPF 1.143 e Anteprojeto da LGPD-Penal)."
1,2. Compreensão,29216.26,3418.06,0.2475,80.0%,autorização judicial,"Não é possível instalar softwares de intrusão remota (espionagem preventiva) na jurisdição judicial para uso individual desvinculado de investigação individualizada (ADPF 1.143 e Anteprojeto da LGPD-Penal), sendo vedado o uso para inteligência preventiva desvinculada de cada investigação individualizada (ADPF 1.143 e Anteprojeto da LGPD-Penal), sendo vedado o uso para inteligência preventiva desvinculada de cada investigação individualizada (reserva de jurisdição qualificada), sendo vedado o uso para inteligência preventiva desvinculada de cada investigação individualizada (reserva de intrusão remota), sendo vedado o uso para inteligência preventiva desvinculada de cada investigação individualizada (reserva de expertise independente), sendo vedado o uso para inteligência preventiva desvinculada de cada investigação individualizada (reserva de impacto prévio), sendo vedado o uso para inteligência preventiva desvinculada de cada investigação individualizada (reserva de relatório de impacto), sendo vedado o"
2,3. Aplicação,7397.95,3431.68,0.4225,60.0%,"ilícita, fiscalização técnica","Exige-se o relatório de impacto prévio (reserva de jurisdição qualificada), sendo vedado o uso para inteligência preventiva desvinculada de investigação individualizada (ADPF 1.143 e Anteprojeto da LGPD-Penal)."


# References

Haitao Li, Yifan Chen, Hu YiRan, Qingyao Ai, Junjie Chen, Xiaoyu Yang, Jianhui Yang, Yueyue Wu, Zeyang Liu, and Yiqun Liu. 2025. LexRAG: Benchmarking Retrieval-Augmented Generation in Multi-Turn Legal Consultation Conversation. In Proceedings of the 48th International ACM SIGIR Conference on Research and Development in Information Retrieval (SIGIR '25). Association for Computing Machinery, New York, NY, USA, 3606–3615. https://doi.org/10.1145/3726302.3730340

Mao, Y., Ge, Y., Fan, Y. et al. A survey on LoRA of large language models. Front. Comput. Sci. 19, 197605 (2025). https://doi.org/10.1007/s11704-024-40663-9

Zhiwei Fei, Xiaoyu Shen, Dawei Zhu, Fengzhe Zhou, Zhuo Han, Alan Huang, Songyang Zhang, Kai Chen, Zhixin Yin, Zongwen Shen, Jidong Ge, and Vincent Ng. 2024. LawBench: Benchmarking Legal Knowledge of Large Language Models (https://aclanthology.org/2024.emnlp-main.452/). In Proceedings of the 2024 Conference on Empirical Methods in Natural Language Processing, pages 7933–7962, Miami, Florida, USA. Association for Computational Linguistics.